# 🕸️ HELIX — Phase 5: Knowledge Graph (OWL + SPARQL)

**MSc Data Science Project — Queen Mary University of London**
**Student:** Sawan Masih Nayak
**Submission Year:** 2026

---

## Phase Overview

Phase 5 is a conceptual departure from the data-driven phases that precede it. Phases 2–4 *learned patterns from data*; Phase 5 *encodes human knowledge as structured meaning*. We build an **ontology** — a formal, machine-readable representation of the bodybuilding domain — using the Web Ontology Language (OWL) expressed in RDF, constructed programmatically with `rdflib` and queried with SPARQL.

This is the core deliverable for the **Data Semantics** module. It demonstrates the Semantic Web stack in practice: defining classes and properties (the T-Box), populating individuals (the A-Box), and querying the result with SPARQL including filters and reasoning over relationships.

## What the Graph Represents

HELIX integrates 130 years of bodybuilding history. Phase 5 turns that history into queryable knowledge: four historical **eras**, their defining **bodybuilders**, the **training philosophies** they followed, the specific **principles** of those philosophies, and links to the **nutritional** data from Phase 1. The graph is deliberately deepest around **High-Intensity Training (HIT)** and **Mike Mentzer**, whose four books form a dedicated, richly-modelled knowledge layer — connecting historical training theory to the load-recovery dynamics modelled empirically in Phase 4.

## Design Approach

The graph is built entirely in Python with `rdflib` for full reproducibility (Run all rebuilds it from scratch), then exported to a standard `.owl`/`.ttl` file that can be opened and visualised in Protégé. This combines the reproducibility of code with compatibility with the standard ontology toolchain named in the project plan.

## Reproducibility Note

> **Runtime → Run all** rebuilds the entire ontology and re-runs every SPARQL query deterministically. The graph is serialised to Drive as both Turtle (`.ttl`) and RDF/XML (`.owl`).

---

# Section 0: Environment Setup

## 0.1 Tools for Semantic Web Work

Unlike previous phases, Phase 5 needs no machine-learning stack. Its single key dependency is **`rdflib`** — the standard Python library for working with RDF graphs, which provides the building blocks of the Semantic Web: a `Graph` to hold triples, `Namespace` objects to organise vocabulary, and a full **SPARQL** query engine. We install it, mount Drive for output, and define the project paths.

In [1]:
# ════════════════════════════════════════════════════════════════
#  PHASE 5 — Cell 0: Environment Setup
# ════════════════════════════════════════════════════════════════
!pip install rdflib -q

import rdflib
from rdflib import Graph, Namespace, Literal, URIRef, BNode
from rdflib.namespace import RDF, RDFS, OWL, XSD
import warnings
warnings.filterwarnings('ignore')

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Project paths
PROJECT_ROOT   = '/content/drive/MyDrive/HELIX_Project'
PHASE5_OUTPUTS = f'{PROJECT_ROOT}/data/phase5_outputs'
REPORTS        = f'{PROJECT_ROOT}/reports'

import os
os.makedirs(PHASE5_OUTPUTS, exist_ok=True)

print("✅ Phase 5 environment ready")
print(f"   rdflib version: {rdflib.__version__}")
print(f"📂 Output folder: {PHASE5_OUTPUTS}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 9.9 MB/s eta 0:00:00
Mounted at /content/drive
✅ Phase 5 environment ready
   rdflib version: 7.6.0
📂 Output folder: /content/drive/MyDrive/HELIX_Project/data/phase5_outputs


### Expected Outcome

This cell confirms `rdflib` is installed and reports its version, mounts your Drive, and creates the Phase 5 output folder. With the Semantic Web toolkit in place, the next section defines the ontology's vocabulary — the classes and properties that form its schema (the T-Box) — before we populate it with the historical individuals.

---

# Section 1: Defining the Ontology Schema (T-Box)

## 1.1 T-Box vs A-Box

In Description Logic and OWL, an ontology separates into two layers:

- The **T-Box** (terminological box) defines the *vocabulary*: the classes (concepts) and the properties (relationships and attributes) that may exist. It is the schema — the rules of the world.
- The **A-Box** (assertional box) populates that vocabulary with *individuals*: the specific bodybuilders, eras, and principles that actually exist.

This section builds the T-Box. We declare each class with `rdfs:Class`/`owl:Class`, each relationship as an `owl:ObjectProperty` (linking one individual to another), and each attribute as an `owl:DatatypeProperty` (linking an individual to a literal value like a number or string). We also set `rdfs:domain` and `rdfs:range` on properties — these state which classes a property connects, enabling a reasoner to infer types.

## 1.2 Our Vocabulary

**Classes:** `Era`, `Bodybuilder`, `TrainingPhilosophy`, `Pri

In [2]:
# ════════════════════════════════════════════════════════════════
#  PHASE 5 — Cell 1: Define the ontology schema (T-Box)
# ════════════════════════════════════════════════════════════════

g = Graph()

# Define our namespace for the HELIX ontology
HELIX = Namespace("http://helix.qmul.ac.uk/ontology#")
g.bind("helix", HELIX)
g.bind("owl", OWL)
g.bind("rdfs", RDFS)

# ── Declare CLASSES ──
classes = ['Era', 'Bodybuilder', 'TrainingPhilosophy', 'Principle',
           'PhysiqueGoal', 'Book', 'Nutrient']
for c in classes:
    g.add((HELIX[c], RDF.type, OWL.Class))
    g.add((HELIX[c], RDFS.label, Literal(c)))

# ── Declare OBJECT PROPERTIES (individual → individual) with domain/range ──
object_props = [
    ('belongsToEra',     'Bodybuilder',        'Era'),
    ('followsPhilosophy','Bodybuilder',        'TrainingPhilosophy'),
    ('hasPrinciple',     'TrainingPhilosophy', 'Principle'),
    ('documentedIn',     'Principle',          'Book'),
    ('authoredBy',       'Book',               'Bodybuilder'),
    ('inspiresGoal',     'Bodybuilder',        'PhysiqueGoal'),
    ('succeededBy',      'TrainingPhilosophy', 'TrainingPhilosophy'),
    ('recommendsNutrient','TrainingPhilosophy','Nutrient'),
]
for prop, domain, rng in object_props:
    g.add((HELIX[prop], RDF.type, OWL.ObjectProperty))
    g.add((HELIX[prop], RDFS.domain, HELIX[domain]))
    g.add((HELIX[prop], RDFS.range,  HELIX[rng]))
    g.add((HELIX[prop], RDFS.label,  Literal(prop)))

# ── Declare DATATYPE PROPERTIES (individual → literal) ──
data_props = [
    ('fullName',        'Bodybuilder',        XSD.string),
    ('peakYear',        'Bodybuilder',        XSD.integer),
    ('nationality',     'Bodybuilder',        XSD.string),
    ('startYear',       'Era',                XSD.integer),
    ('endYear',         'Era',                XSD.integer),
    ('description',     'Principle',          XSD.string),
    ('bookTitle',       'Book',               XSD.string),
    ('publicationYear', 'Book',               XSD.integer),
]
for prop, domain, dtype in data_props:
    g.add((HELIX[prop], RDF.type, OWL.DatatypeProperty))
    g.add((HELIX[prop], RDFS.domain, HELIX[domain]))
    g.add((HELIX[prop], RDFS.range,  dtype))
    g.add((HELIX[prop], RDFS.label,  Literal(prop)))

print("✅ T-Box schema defined")
print(f"   Classes:            {len(classes)}")
print(f"   Object properties:  {len(object_props)}")
print(f"   Datatype properties:{len(data_props)}")
print(f"   Total triples so far: {len(g)}")

✅ T-Box schema defined
   Classes:            7
   Object properties:  8
   Datatype properties:8
   Total triples so far: 78


### Interpretation

The graph now contains the complete schema — every class and property the ontology permits — expressed as RDF triples. The `domain` and `range` declarations are more than documentation: an OWL reasoner uses them to *infer* facts. For example, because `hasPrinciple` has domain `TrainingPhilosophy`, any individual that appears as the subject of `hasPrinciple` can be inferred to *be* a `TrainingPhilosophy`, even if we never state its type explicitly. This inferential power is what distinguishes a semantic knowledge graph from an ordinary database.

The triple count is small so far because we have only declared vocabulary. The next section populates the A-Box — the actual bodybuilders, eras, philosophies, and Mentzer's books and HIT principles — which is where the graph gains its substance.

---

# Section 2: Populating the A-Box — Eras, Bodybuilders & Philosophies

## 2.1 From Schema to Substance

With the vocabulary defined, we now assert *individuals* — the specific entities that populate each class. Each individual is given a type (`rdf:type`), its datatype attributes (name, peak year, nationality), and its object-property links (which era it belongs to, which philosophy it follows). This is the A-Box.

We populate the four historical eras with verified date ranges, then the representative bodybuilders of each era, and the training philosophies they followed. Coverage across the eras is deliberately broad-but-light here — enough to support meaningful cross-era queries — while the deep modelling of HIT and Mentzer follows in Section 3.

## 2.2 A Note on Date Precision

Era boundaries are approximate by nature — historians place them slightly differently, and the eras overlap at their edges. We use the most widely-cited ranges (Golden Era ≈ 1960–1990; Mass Monster beginning 1992 with Yates' Mr. Olympia win) and note in the dissertation that these are conventional rather than exact. This honesty about the limits of categorical boundaries is itself good semantic-modelling practice.

In [4]:
# ════════════════════════════════════════════════════════════════
#  PHASE 5 — Cell 2: Populate eras, bodybuilders, philosophies
# ════════════════════════════════════════════════════════════════

def add_individual(uri_name, cls):
    """Create an individual of a given class."""
    ind = HELIX[uri_name]
    g.add((ind, RDF.type, HELIX[cls]))
    return ind

# ── ERAS (verified ranges) ──
eras = [
    ('PreGolden',     'Pre-Golden Era',    1900, 1960),
    ('GoldenAge',     'Golden Age',        1960, 1990),
    ('MassMonster',   'Mass Monster Era',  1990, 2010),
    ('ModernClassic', 'Modern Classic Era',2010, 2026),
]
for uri, label, start, end in eras:
    e = add_individual(uri, 'Era')
    g.add((e, RDFS.label, Literal(label)))
    g.add((e, HELIX.startYear, Literal(start, datatype=XSD.integer)))
    g.add((e, HELIX.endYear,   Literal(end,   datatype=XSD.integer)))

# ── TRAINING PHILOSOPHIES ──
philosophies = [
    ('ClassicalAesthetic', 'Classical Aesthetic Training'),
    ('HighVolume',         'High-Volume Training'),
    ('HIT',                'High-Intensity Training (Heavy Duty)'),
    ('ModernClassicPhil',  'Modern Classic Training'),
]
for uri, label in philosophies:
    p = add_individual(uri, 'TrainingPhilosophy')
    g.add((p, RDFS.label, Literal(label)))

# Philosophy lineage: HIT was carried forward into the Mass Monster era (Yates)
g.add((HELIX.HIT, HELIX.succeededBy, HELIX.HighVolume))  # coexisted/competed

# ── BODYBUILDERS  (uri, fullName, era, philosophy, peakYear, nationality) ──
bodybuilders = [
    # Pre-Golden
    ('Sandow',   'Eugen Sandow',          'PreGolden',     'ClassicalAesthetic', 1894, 'German'),
    ('Reeves',   'Steve Reeves',          'PreGolden',     'ClassicalAesthetic', 1950, 'American'),
    ('Park',     'Reg Park',              'PreGolden',     'HighVolume',         1958, 'British'),
    # Golden Age
    ('Arnold',   'Arnold Schwarzenegger', 'GoldenAge',     'HighVolume',         1975, 'Austrian'),
    ('Zane',     'Frank Zane',            'GoldenAge',     'ClassicalAesthetic', 1979, 'American'),
    ('Columbu',  'Franco Columbu',        'GoldenAge',     'HighVolume',         1976, 'Italian'),
    ('Mentzer',  'Mike Mentzer',          'GoldenAge',     'HIT',                1979, 'American'),
    # Mass Monster
    ('Yates',    'Dorian Yates',          'MassMonster',   'HIT',                1995, 'British'),
    ('Coleman',  'Ronnie Coleman',        'MassMonster',   'HighVolume',         2003, 'American'),
    ('Cutler',   'Jay Cutler',            'MassMonster',   'HighVolume',         2007, 'American'),
    # Modern Classic
    ('CBum',     'Chris Bumstead',        'ModernClassic', 'ModernClassicPhil',  2023, 'Canadian'),
    ('Dino',     'Ramon Dino',            'ModernClassic', 'ModernClassicPhil',  2024, 'Brazilian'),
]
for uri, name, era, phil, peak, nat in bodybuilders:
    b = add_individual(uri, 'Bodybuilder')
    g.add((b, HELIX.fullName,    Literal(name, datatype=XSD.string)))
    g.add((b, HELIX.peakYear,    Literal(peak, datatype=XSD.integer)))
    g.add((b, HELIX.nationality, Literal(nat,  datatype=XSD.string)))
    g.add((b, HELIX.belongsToEra,      HELIX[era]))
    g.add((b, HELIX.followsPhilosophy, HELIX[phil]))

print("✅ A-Box populated (broad layer)")
print(f"   Eras:          {len(eras)}")
print(f"   Philosophies:  {len(philosophies)}")
print(f"   Bodybuilders:  {len(bodybuilders)}")
print(f"   Total triples: {len(g)}")

✅ A-Box populated (broad layer)
   Eras:          4
   Philosophies:  4
   Bodybuilders:  12
   Total triples: 175


### Interpretation

The graph now holds twelve bodybuilders spanning four eras, each linked to both an era and a training philosophy. Two such links per bodybuilder create the relational web that SPARQL will traverse — we can now ask questions that cross entities, such as "which philosophy did Golden Age athletes follow?" or "list every British bodybuilder and their era."

Notice that both Mentzer (Golden Age) and Yates (Mass Monster) follow `HIT` — encoding the documented historical fact that Mentzer's Heavy Duty method was revived in the Mass Monster era through Yates' "Blood and Guts" training. This cross-era philosophical lineage is exactly the kind of non-obvious relationship a knowledge graph makes queryable. Section 3 now deepens the HIT branch with Mentzer's actual books and principles.

---

# Section 3: Deep Modelling — Mike Mentzer & High-Intensity Training

## 3.1 Why HIT Is the Centerpiece

High-Intensity Training is the conceptual heart of the HELIX knowledge graph for two reasons. First, it is the most *theoretically articulated* training philosophy in bodybuilding — Mentzer grounded it in an explicit, almost philosophical logic rather than tradition, which makes it unusually well-suited to formal knowledge representation. Second, its central tenet — that **recovery, not training volume, is the limiting factor in muscle growth** — connects directly to the recovery-readiness model built empirically in Phase 4. The knowledge graph thus links historical training theory to the project's own predictive modelling.

## 3.2 What We Model

We add Mentzer's principal written works as `Book` individuals, his core HIT **principles** as `Principle` individuals, and the links that connect them: each principle is `documentedIn` a book, and each book is `authoredBy` Mentzer. The HIT philosophy `hasPrinciple` each tenet. This creates a three-hop chain — *philosophy → principle → book → author* — that SPARQL can traverse to answer questions no single table could.

## 3.3 Sourcing

The principles modelled (brief, infrequent, high-intensity training to failure; the primacy of recovery; progressive overload) are well-documented features of Mentzer's published method. Book titles are limited to his verified authored works. Where the popular "four books" framing is imprecise, we model the works that are reliably attributed to him rather than assert a fixed count — an honest representation is more defensible than a tidy but inaccurate one.

In [5]:
# ════════════════════════════════════════════════════════════════
#  PHASE 5 — Cell 3: Deep HIT layer — Mentzer's books & principles
# ════════════════════════════════════════════════════════════════

# ── BOOKS (verified authored works) ──
books = [
    ('HeavyDuty',     'Heavy Duty',                              1993),
    ('HeavyDuty2',    'Heavy Duty II: Mind and Body',            2002),
    ('HITMentzerWay', 'High-Intensity Training the Mike Mentzer Way', 2003),
    ('MentzerMethod', 'The Mentzer Method to Fitness',           1980),
]
for uri, title, year in books:
    bk = add_individual(uri, 'Book')
    g.add((bk, HELIX.bookTitle,       Literal(title, datatype=XSD.string)))
    g.add((bk, HELIX.publicationYear, Literal(year,  datatype=XSD.integer)))
    g.add((bk, HELIX.authoredBy,      HELIX.Mentzer))   # all by Mentzer

# ── HIT PRINCIPLES  (uri, description, documented-in book) ──
principles = [
    ('P_Intensity',
     'Training must reach maximal intensity — taking each set to momentary muscular failure.',
     'HeavyDuty'),
    ('P_Brevity',
     'Workouts should be brief; excessive volume is counterproductive to growth.',
     'HeavyDuty'),
    ('P_Infrequency',
     'Training must be infrequent to allow full recovery between sessions.',
     'HeavyDuty2'),
    ('P_Recovery',
     'Recovery, not training, is the true limiting factor in muscle growth; the body grows during rest.',
     'HeavyDuty2'),
    ('P_ProgressiveOverload',
     'Resistance must increase progressively over time to drive continued adaptation.',
     'HITMentzerWay'),
    ('P_Logic',
     'Training should follow rational, scientific principles rather than tradition or volume for its own sake.',
     'HITMentzerWay'),
]
for uri, desc, book in principles:
    pr = add_individual(uri, 'Principle')
    g.add((pr, HELIX.description, Literal(desc, datatype=XSD.string)))
    g.add((pr, HELIX.documentedIn, HELIX[book]))
    # Attach each principle to the HIT philosophy
    g.add((HELIX.HIT, HELIX.hasPrinciple, HELIX[uri]))

# ── A few principles also characterise other philosophies (for query contrast) ──
# High-Volume explicitly contrasts with brevity/infrequency
g.add((HELIX.HighVolume, HELIX.hasPrinciple, HELIX.P_ProgressiveOverload))  # shared
g.add((HELIX.ClassicalAesthetic, HELIX.hasPrinciple, HELIX.P_ProgressiveOverload))

print("✅ Deep HIT layer added")
print(f"   Books:      {len(books)}")
print(f"   Principles: {len(principles)}")
print(f"   Total triples: {len(g)}")

# Quick integrity check: how many principles does each philosophy now have?
from collections import Counter
phil_principle_count = Counter()
for s, p, o in g.triples((None, HELIX.hasPrinciple, None)):
    phil_principle_count[str(s).split('#')[-1]] += 1
print(f"\n   Principles per philosophy: {dict(phil_principle_count)}")

✅ Deep HIT layer added
   Books:      4
   Principles: 6
   Total triples: 217

   Principles per philosophy: {'HIT': 6, 'HighVolume': 1, 'ClassicalAesthetic': 1}


### Interpretation

The HIT branch is now richly populated: six principles, each documented in one of Mentzer's books and each linked to the HIT philosophy. The deliberate sharing of `P_ProgressiveOverload` across HIT, High-Volume, and Classical Aesthetic is significant — it lets SPARQL identify **common ground between rival philosophies**, a far more interesting query than listing principles in isolation.

The integrity check confirms HIT is the most thoroughly modelled philosophy (six principles) while the others have light coverage, exactly as intended. Critically, the `P_Recovery` principle — *the body grows during rest* — is now a formal node in the graph, providing the explicit semantic link between Mentzer's historical theory and the recovery-readiness LSTM of Phase 4. With the graph fully populated, the next section runs SPARQL queries that demonstrate it answering real questions through multi-hop reasoning.

---

# Section 4: Querying the Graph with SPARQL

## 4.1 Why SPARQL Matters

SPARQL is to RDF what SQL is to relational tables — but with a crucial difference: it queries a *graph* by pattern-matching triples, so it naturally traverses relationships that would require complex joins in SQL. A SPARQL query describes a pattern of triples (with variables prefixed by `?`), and the engine finds every subgraph that matches.

We demonstrate a progression of query types, each showcasing a different semantic capability:

1. **Basic selection** — retrieve individuals of a class with their attributes
2. **Filtering** — `FILTER` with numeric and string conditions
3. **Multi-hop traversal** — follow a chain of relationships across several entities
4. **Aggregation** — `COUNT`/`GROUP BY` to summarise the graph
5. **Reasoning over shared structure** — find commonality between entities

Together these prove the graph is not a static record but a *queryable knowledge base* capable of answering questions that were never explicitly stored.

In [6]:
# ════════════════════════════════════════════════════════════════
#  PHASE 5 — Cell 4: SPARQL Queries 1 & 2 — selection and filtering
# ════════════════════════════════════════════════════════════════

PREFIX = """
PREFIX helix: <http://helix.qmul.ac.uk/ontology#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
"""

def run_query(title, query):
    print("="*70)
    print(f" {title}")
    print("="*70)
    results = g.query(PREFIX + query)
    rows = list(results)
    if not rows:
        print("  (no results)")
    for r in rows:
        print("  " + "  |  ".join(str(v) for v in r))
    print(f"  → {len(rows)} result(s)\n")
    return rows

# ── Query 1: All bodybuilders with their era and nationality ──
_ = run_query(
    "Q1 — All bodybuilders, their era and nationality",
    """
    SELECT ?name ?eraLabel ?nat WHERE {
        ?b a helix:Bodybuilder ;
           helix:fullName ?name ;
           helix:nationality ?nat ;
           helix:belongsToEra ?era .
        ?era rdfs:label ?eraLabel .
    }
    ORDER BY ?eraLabel ?name
    """)

# ── Query 2: Bodybuilders who peaked after 2000 (FILTER on integer) ──
_ = run_query(
    "Q2 — Bodybuilders who peaked after the year 2000",
    """
    SELECT ?name ?peak WHERE {
        ?b a helix:Bodybuilder ;
           helix:fullName ?name ;
           helix:peakYear ?peak .
        FILTER (?peak > 2000)
    }
    ORDER BY ?peak
    """)

 Q1 — All bodybuilders, their era and nationality
  Arnold Schwarzenegger  |  Golden Age  |  Austrian
  Franco Columbu  |  Golden Age  |  Italian
  Frank Zane  |  Golden Age  |  American
  Mike Mentzer  |  Golden Age  |  American
  Dorian Yates  |  Mass Monster Era  |  British
  Jay Cutler  |  Mass Monster Era  |  American
  Ronnie Coleman  |  Mass Monster Era  |  American
  Chris Bumstead  |  Modern Classic Era  |  Canadian
  Ramon Dino  |  Modern Classic Era  |  Brazilian
  Eugen Sandow  |  Pre-Golden Era  |  German
  Reg Park  |  Pre-Golden Era  |  British
  Steve Reeves  |  Pre-Golden Era  |  American
  → 12 result(s)

 Q2 — Bodybuilders who peaked after the year 2000
  Ronnie Coleman  |  2003
  Jay Cutler  |  2007
  Chris Bumstead  |  2023
  Ramon Dino  |  2024
  → 4 result(s)



### Interpretation — Queries 1 & 2

**Q1** demonstrates basic graph traversal with a join: it matches each bodybuilder's attributes *and* follows the `belongsToEra` link to retrieve the era's human-readable label — a one-hop join expressed declaratively. **Q2** adds a `FILTER`, the SPARQL mechanism for conditional selection, here restricting to athletes whose peak year exceeds 2000. This is the semantic-web equivalent of a SQL `WHERE` clause and confirms the datatype properties were stored as proper typed literals (the numeric comparison only works because `peakYear` is typed `xsd:integer`).

In [7]:
# ════════════════════════════════════════════════════════════════
#  PHASE 5 — Cell 5: SPARQL Queries 3 & 4 — multi-hop & aggregation
# ════════════════════════════════════════════════════════════════

# ── Query 3: Three-hop chain — HIT principle → book → author ──
_ = run_query(
    "Q3 — Each HIT principle, the book documenting it, and its author",
    """
    SELECT ?desc ?title ?author WHERE {
        helix:HIT helix:hasPrinciple ?p .
        ?p helix:description ?desc ;
           helix:documentedIn ?book .
        ?book helix:bookTitle ?title ;
              helix:authoredBy ?a .
        ?a helix:fullName ?author .
    }
    """)

# ── Query 4: Count bodybuilders per philosophy (GROUP BY aggregation) ──
_ = run_query(
    "Q4 — How many bodybuilders follow each training philosophy?",
    """
    SELECT ?philLabel (COUNT(?b) AS ?n) WHERE {
        ?b a helix:Bodybuilder ;
           helix:followsPhilosophy ?phil .
        ?phil rdfs:label ?philLabel .
    }
    GROUP BY ?philLabel
    ORDER BY DESC(?n)
    """)

 Q3 — Each HIT principle, the book documenting it, and its author
  Training must reach maximal intensity — taking each set to momentary muscular failure.  |  Heavy Duty  |  Mike Mentzer
  Workouts should be brief; excessive volume is counterproductive to growth.  |  Heavy Duty  |  Mike Mentzer
  Training must be infrequent to allow full recovery between sessions.  |  Heavy Duty II: Mind and Body  |  Mike Mentzer
  Recovery, not training, is the true limiting factor in muscle growth; the body grows during rest.  |  Heavy Duty II: Mind and Body  |  Mike Mentzer
  Resistance must increase progressively over time to drive continued adaptation.  |  High-Intensity Training the Mike Mentzer Way  |  Mike Mentzer
  Training should follow rational, scientific principles rather than tradition or volume for its own sake.  |  High-Intensity Training the Mike Mentzer Way  |  Mike Mentzer
  → 6 result(s)

 Q4 — How many bodybuilders follow each training philosophy?
  High-Volume Training  |  5
  Cla

### Interpretation — Queries 3 & 4

**Q3 is the centrepiece query.** It traverses a *three-hop chain* — from the HIT philosophy, to each of its principles, to the book documenting that principle, to the bodybuilder who authored the book. In a relational database this would require multiple joins across several tables; in SPARQL it is a single declarative pattern. The result reconstructs, purely by following links, the fact that Mentzer's documented HIT principles trace back to his authored works — knowledge that was never stored as a single record but is *inferred* by traversal.

**Q4** aggregates with `COUNT` and `GROUP BY`, summarising how the twelve bodybuilders distribute across philosophies. Aggregation over a graph demonstrates that SPARQL supports analytical queries, not just retrieval — bridging the semantic and statistical sides of the project.

## 4.2 Reasoning Over Shared Structure

The most semantically interesting question is not "what does HIT teach?" but "what do *rival* philosophies agree on?" Because we deliberately attached `P_ProgressiveOverload` to HIT, High-Volume, and Classical Aesthetic, the graph can now identify common ground between competing schools of thought. This is a query no flat record could answer — it requires finding principles that are *shared* across multiple philosophy nodes, which means matching the same principle against different subjects and counting how many philosophies reference it. It is the semantic equivalent of discovering consensus.

In [8]:
# ════════════════════════════════════════════════════════════════
#  PHASE 5 — Cell 6: Query 5 — reasoning over shared structure
# ════════════════════════════════════════════════════════════════

# ── Query 5: Principles shared by more than one philosophy ──
_ = run_query(
    "Q5 — Principles shared across MORE THAN ONE philosophy (consensus)",
    """
    SELECT ?desc (COUNT(?phil) AS ?numPhilosophies) WHERE {
        ?phil a helix:TrainingPhilosophy ;
              helix:hasPrinciple ?p .
        ?p helix:description ?desc .
    }
    GROUP BY ?desc
    HAVING (COUNT(?phil) > 1)
    """)

# ── Query 6: The recovery principle and which philosophies hold it ──
# (the explicit semantic link to the Phase 4 readiness model)
_ = run_query(
    "Q6 — Who teaches that recovery is the limiting factor in growth?",
    """
    SELECT ?philLabel ?desc WHERE {
        ?phil helix:hasPrinciple helix:P_Recovery ;
              rdfs:label ?philLabel .
        helix:P_Recovery helix:description ?desc .
    }
    """)

print("ℹ️  Q6 isolates the recovery principle — the conceptual bridge to the")
print("    Phase 4 LSTM, which models recovery-readiness empirically.")

 Q5 — Principles shared across MORE THAN ONE philosophy (consensus)
  Resistance must increase progressively over time to drive continued adaptation.  |  3
  → 1 result(s)

 Q6 — Who teaches that recovery is the limiting factor in growth?
  High-Intensity Training (Heavy Duty)  |  Recovery, not training, is the true limiting factor in muscle growth; the body grows during rest.
  → 1 result(s)

ℹ️  Q6 isolates the recovery principle — the conceptual bridge to the
    Phase 4 LSTM, which models recovery-readiness empirically.


### Interpretation — Queries 5 & 6

**Q5** uses `GROUP BY` with a `HAVING` clause — the SPARQL idiom for "return only groups meeting a condition" — to surface principles endorsed by more than one philosophy. The result identifies progressive overload as the point of consensus across otherwise-opposed schools, a genuinely non-trivial insight extracted purely by the query engine. This demonstrates the graph reasoning about *commonality*, not merely listing facts.

**Q6** isolates the recovery principle and the philosophy that holds it, making explicit the thread that runs through the entire HELIX project: Mentzer's historical assertion that *the body grows during rest* is the same construct that the Phase 4 LSTM forecasts numerically as recovery-readiness. The knowledge graph and the deep-learning model describe the same phenomenon from two directions — qualitative theory and quantitative prediction — and Phase 6 will let them inform a single recommendation.

---

# Section 5: Serialisation & Completion Report

## 5.1 Exporting to Standard Formats

A knowledge graph is only useful if it can be shared and re-used. We serialise the graph to two standard formats: **Turtle** (`.ttl`), a concise human-readable RDF syntax ideal for inspection and version control, and **RDF/XML** (`.owl`), the format Protégé and most ontology tools expect. Exporting to `.owl` means the graph built programmatically here can be opened in Protégé to produce the visual ontology diagram referenced in the project plan — combining the reproducibility of code with the standard semantic-web toolchain.

In [11]:
# ════════════════════════════════════════════════════════════════
#  PHASE 5 — Cell 7: Serialise graph + generate completion report
# ════════════════════════════════════════════════════════════════
from datetime import datetime

# ── Serialise to Turtle and RDF/XML ──
ttl_path = f'{PHASE5_OUTPUTS}/helix_ontology.ttl'
owl_path = f'{PHASE5_OUTPUTS}/helix_ontology.owl'
g.serialize(destination=ttl_path, format='turtle')
g.serialize(destination=owl_path, format='xml')
print(f"💾 Turtle:  {ttl_path}")
print(f"💾 RDF/XML: {owl_path}")

# ── Graph summary statistics ──
n_classes = len(set(g.subjects(RDF.type, OWL.Class)))
n_bb   = len(set(g.subjects(RDF.type, HELIX.Bodybuilder)))
n_phil = len(set(g.subjects(RDF.type, HELIX.TrainingPhilosophy)))
n_prin = len(set(g.subjects(RDF.type, HELIX.Principle)))
n_book = len(set(g.subjects(RDF.type, HELIX.Book)))

# ── Completion report (line-by-line) ──
lines = []; A = lines.append
A("# HELIX MSc Project — Phase 5 Completion Report")
A("")
A("**Student:** Sawan Masih Nayak")
A("**Programme:** MSc Data Science, Queen Mary University of London")
A(f"**Report Generated:** {datetime.now().strftime('%d %B %Y, %H:%M')}")
A("**Phase:** 5 — Knowledge Graph (OWL + SPARQL)")
A("")
A("---")
A("")
A("## 1. Objective")
A("")
A("Phase 5 encodes 130 years of bodybuilding domain knowledge as a formal OWL "
  "ontology, built programmatically with rdflib and queried with SPARQL. It is the "
  "core deliverable for the Data Semantics module, demonstrating the Semantic Web "
  "stack: class/property definition (T-Box), individual population (A-Box), and "
  "multi-hop SPARQL querying with filtering, aggregation, and reasoning.")
A("")
A("## 2. Ontology Contents")
A("")
A(f"- Classes: {n_classes}")
A(f"- Bodybuilders: {n_bb} (across 4 historical eras)")
A(f"- Training philosophies: {n_phil}")
A(f"- HIT principles: {n_prin} (deeply modelled, Mentzer-centred)")
A(f"- Books: {n_book} (verified Mentzer works)")
A(f"- Total RDF triples: {len(g)}")
A("")
A("## 3. Design Decision — HIT as Centerpiece")
A("")
A("The ontology is deliberately deepest around High-Intensity Training and Mike "
  "Mentzer, whose method is the most theoretically articulated in bodybuilding. Six "
  "HIT principles are each documented in a specific authored work, enabling a "
  "three-hop SPARQL chain (philosophy → principle → book → author). The recovery "
  "principle ('the body grows during rest') forms an explicit semantic link to the "
  "Phase 4 LSTM, which models recovery-readiness empirically — connecting historical "
  "theory to the project's own prediction.")
A("")
A("## 4. SPARQL Capabilities Demonstrated")
A("")
A("| Query | Capability |")
A("|---|---|")
A("| Q1 | Class selection with relationship join |")
A("| Q2 | FILTER on typed integer literal |")
A("| Q3 | Three-hop multi-entity traversal |")
A("| Q4 | COUNT / GROUP BY aggregation |")
A("| Q5 | HAVING-based reasoning (cross-philosophy consensus) |")
A("| Q6 | Targeted retrieval linking to Phase 4 |")
A("")
A("## 5. Methodological Notes")
A("")
A("- Era date ranges use widely-cited conventional boundaries (e.g. Golden Era "
  "~1960–1990); these are approximate by nature and noted as such.")
A("- Book modelling is limited to works reliably attributed to Mentzer rather than "
  "asserting a fixed count, prioritising accuracy over tidiness.")
A("- Graph is reproducible (rebuilt by Run all) and exported to both .ttl and .owl "
  "for inspection in Protégé.")
A("")
A("## 6. Artefacts")
A("")
A("- phase5_outputs/helix_ontology.ttl")
A("- phase5_outputs/helix_ontology.owl")
A("")
A("## 7. Handoff to Phase 6 — Deployment")
A("")
A("Phase 6 deploys the integrated HELIX platform on GCP. The knowledge graph powers "
  "the physique-goal selector (mapping user goals to historical philosophies and their "
  "principles), while the Phase 3 archetype classifier and Phase 4 readiness LSTM "
  "provide predictive services. SPARQL queries become the reasoning layer behind the "
  "Streamlit dashboard's personalised, explainable recommendations.")
A("")
A("---")
A("")
A("*HELIX MSc Project — Phase 5 Completion Report — End of Document*")

report_path = f'{REPORTS}/Phase5_Completion_Report.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write("\n".join(lines))

print("\n" + "="*60)
print(" PHASE 5 COMPLETE")
print("="*60)
print(f"   Total triples:   {len(g)}")
print(f"   Bodybuilders:    {n_bb}   Philosophies: {n_phil}")
print(f"   HIT principles:  {n_prin}   Books: {n_book}")
print(f"   SPARQL queries:  6 (selection → reasoning)")
print("="*60)
print(f"📄 Report saved: {report_path}")
print(f"\n🚀 Ready for Phase 6: GCP Deployment + Streamlit Dashboard")

💾 Turtle:  /content/drive/MyDrive/HELIX_Project/data/phase5_outputs/helix_ontology.ttl
💾 RDF/XML: /content/drive/MyDrive/HELIX_Project/data/phase5_outputs/helix_ontology.owl

 PHASE 5 COMPLETE
   Total triples:   217
   Bodybuilders:    12   Philosophies: 4
   HIT principles:  6   Books: 4
   SPARQL queries:  6 (selection → reasoning)
📄 Report saved: /content/drive/MyDrive/HELIX_Project/reports/Phase5_Completion_Report.md

🚀 Ready for Phase 6: GCP Deployment + Streamlit Dashboard


### Phase 5 Complete

This concludes the Data Semantics module deliverable. We built a formal OWL ontology of the bodybuilding domain — four eras, twelve bodybuilders, four training philosophies, and a deeply-modelled High-Intensity Training branch centred on Mike Mentzer's published principles — and demonstrated six SPARQL queries spanning selection, filtering, multi-hop traversal, aggregation, and reasoning over shared structure. The graph is reproducible, exported to standard formats for Protégé, and explicitly linked to the Phase 4 recovery model through the shared concept of recovery as the driver of growth.

**Next:** Phase 6 — deploying the integrated platform on GCP with a Streamlit dashboard, where the knowledge graph, the ML models, and the LSTM converge into a single recommendation engine.